# Dataset folder structure
```
data/
│
├── images/                              # PNG files
│   ├── 0.png                           
│   │   .
│   │   .
│   │   .
│   └── 1.png                          
│
└── labels/                                # txt files
	├── 0.txt
	.
	.
	.
	└── 1.txt
```

In [ ]:
from pathlib import Path

MODEL='ETLTC F'

EPOCHS=1

LR=5e-6     #1e-4

DATA_PATH = Path('data/kkanji2')

USE_CHECKPOINT = True

DATASET_PATH = Path('preprocessed_data/kkanji2')

K49_PATH = Path('preprocessed_data/K49')

DATA_PATH.mkdir(parents=True, exist_ok=True)

DATASET_PATH.mkdir(parents=True, exist_ok=True)

PREPROCESS_DATA = False

AUGMENT_DATA = False

# Build lists of images and texts

In [ ]:
import shutil
if PREPROCESS_DATA:
    shutil.unpack_archive('../kmnist/kkanji.tar', 'data')

In [ ]:
import glob
import os
from PIL import Image

if PREPROCESS_DATA:
    word_files = sorted(glob.glob(str(DATA_PATH / '**' / '*.png'), recursive=True))

    print(f"{len(word_files)} word folders")
    print(word_files[0])

    data_dict = {}
    
    vocab = set()
    with open('google_sp.vocab', 'r') as f:
        for line in f.readlines():
            vocab.add(line[0])
            
    vocab = list(vocab)
    print(vocab[100:120])
    print(len(vocab))

    for word_file in word_files:
        filename = os.path.basename(word_file)
        unicode = os.path.basename(os.path.dirname(word_file))
        hex = unicode[2:]
        label = chr(int(hex, 16))
        if label in vocab:
            savepath = str(DATASET_PATH / filename)
            data_dict[savepath] = label
            image = Image.open(word_file)
            image.save(savepath)
            
    print(len(data_dict))
    
    
    


In [ ]:
import json
from util.augmentation import augment_dataset
from util.data_processing import save_labels, split_dataset

if PREPROCESS_DATA:
    save_labels(data_dict, DATASET_PATH / 'labels.json')
    split_dataset(DATASET_PATH)

with open(str(DATASET_PATH / 'train_labels.json'), 'r') as fp:
    data_dict = json.load(fp)

(DATASET_PATH/'augmented').mkdir(parents=True, exist_ok=True)
if AUGMENT_DATA:
    augment_dataset(data_dict, 3, 3, 3)
    save_labels(data_dict, DATASET_PATH / 'augmented_labels.json')

In [ ]:
import json
from pathlib import Path

with open(str(DATASET_PATH / 'augmented_labels.json'), 'r') as fp:
    data_dict = json.load(fp)

print(f"{len(data_dict.items())} dict elements")

with open(str(K49_PATH / 'augmented_labels.json'), 'r') as fp:
    k49_dict = json.load(fp)

print(f"{len(k49_dict.items())} K49 dict elements")

word_files = list(data_dict.items()) + list(k49_dict.items())
print(word_files[0])

In [ ]:
from util.data_processing import get_words_list

with open(str(DATASET_PATH / 'test_labels.json'), 'r') as fp:
    test_dict = json.load(fp)

with open(str(K49_PATH / 'test_labels.json'), 'r') as fp:
    k49_test_dict = json.load(fp)
    
test_words_files = list(test_dict.items()) + list(k49_test_dict.items())
test_words = get_words_list(test_words_files)
train_words = get_words_list(word_files)
print(f'Train size: {len(test_words)}; Test size: {len(train_words)}')

# Build dataset and dataloader

In [ ]:
from util.data_processing import WORDSDataset
from dtrocr.config import DTrOCRConfig

config = DTrOCRConfig(
    # attn_implementation='flash_attention_2'
)
train_data = WORDSDataset(words=train_words, config=config)
test_data = WORDSDataset(words=test_words, config=config)

In [ ]:
from torch.utils.data import DataLoader
import multiprocessing as mp
from util.data_processing import WordsSampler

i = 0

if USE_CHECKPOINT and Path(f'{MODEL} train_samp_ind.txt').exists():
    with open(f'{MODEL} train_samp_ind.txt', 'r') as f:
        i = int(f.readline())
        print(i)
sampler = WordsSampler(train_data, i = i)
print(sampler.seq[0])


    

train_dataloader = DataLoader(train_data, batch_size=32, sampler=sampler, shuffle=False, num_workers=mp.cpu_count())
test_dataloader = DataLoader(test_data, batch_size=32, shuffle=False, num_workers=mp.cpu_count())

In [ ]:
for inputs in test_dataloader:
    print(inputs['labels'])
    break

# Model

In [ ]:
from util.model import get_model

model = get_model(config, MODEL)

# Training

In [ ]:
from util.model import train

train(model, train_dataloader, test_dataloader, EPOCHS, MODEL, LR, 'Kkanji')

# Test

In [ ]:
from dtrocr.model import DTrOCRLMHeadModel
from dtrocr.config import DTrOCRConfig
from dtrocr.processor import DTrOCRProcessor
import torch
torch.save(model.state_dict(), f'../models/{MODEL}.pt')
# model = DTrOCRLMHeadModel(DTrOCRConfig())
model.eval()
model.to('cpu')
test_processor = DTrOCRProcessor(DTrOCRConfig())

In [ ]:
from util.model import test

test(model, test_words, 10)